# Azure Event Grid Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_event_grid/event_grid_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_event_grid/event_grid_demo.ipynb)

## Business Scenario

Azure services emit events when files arrive or devices publish telemetry. You need to ingest those events and validate them immediately.

## Value Proposition

- Event-driven ingestion with schema validation
- Consistent processing for Azure-native events
- Clean handoff to downstream analytics

---

## Goals

1. Connect to Event Grid
2. Validate incoming events
3. Write clean events to storage


## 🚀 Step 1: Setup Azure Event Grid

Before running this notebook, you need:
1. An Azure Event Grid topic or system topic (e.g., Storage Account events)
2. An Event Grid subscription
3. The endpoint URL and subscription name

Set your environment variable:
```bash
export AZURE_EVENT_GRID_ENDPOINT="https://your-topic.region.eventgrid.azure.net/api/events"
```

## 📝 Step 2: Review the Contract

Our contract defines the expected Event Grid schema and quality rules.

In [ ]:
with open('event_grid_contract.yaml', 'r') as f:
    print("📄 Event Grid Contract:")
    print("----------------------")
    print(f.read())

## ▶️ Step 3: Start the Event Grid Listener

This will connect to your Event Grid subscription and start processing events.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(
    contract="event_grid_contract.yaml",
    framework="bytewax"
)

# Start in background
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Event Grid listener started!")
print("Waiting for storage events...")
time.sleep(5)

## 🧪 Step 4: Trigger Test Events

Upload a file to your Azure Storage Account to trigger a `BlobCreated` event.

You should see the event appear in the LakeLogic logs and get materialized to Delta Lake.

In [ ]:
# Optional: Upload a test file using Azure SDK
from azure.storage.blob import BlobServiceClient
import os

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
if connection_string:
    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container = blob_service.get_container_client("test-container")
    
    # Upload a test file
    blob_client = container.get_blob_client("test-file.txt")
    blob_client.upload_blob("Hello from LakeLogic!", overwrite=True)
    
    print("✅ Test file uploaded! Check Event Grid logs...")
else:
    print("ℹ️  Set AZURE_STORAGE_CONNECTION_STRING to test file uploads")

## 🎉 Summary

You just:
- ✅ Connected to Azure Event Grid
- ✅ Validated storage events against a contract
- ✅ Materialized events to Delta Lake

This pattern enables **event-driven data pipelines** where your lakehouse automatically reacts to infrastructure changes!